In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

In [2]:
N_SVD_COMPONENTS = 50
RANDOM_STATE = 42

TFIDF_MAX_FEATURES = 10000
TFIDF_MIN_DF = 3
TFIDF_MAX_DF = 0.85
TFIDF_NGRAM_RANGE = (1, 2)

TEXT_COLUMN_PRIORITY = [
    "plot_clean",
    "plot_final",
    "plotMedium",
    "plotShort",
]

In [3]:
MANUAL_PROJECT_ROOT = None

EXPECTED_FILES = [
    "Lumiere_Country_Full_Backup.csv",
    "clustered_films_k6.csv",
]


def find_project_root(start_path):
    candidates = [start_path] + list(start_path.parents)

    for candidate in candidates:
        data_dir = candidate / "analysis" / "data"
        if all((data_dir / file_name).exists() for file_name in EXPECTED_FILES):
            return candidate

    raise FileNotFoundError(
        "Could not find project root. Set MANUAL_PROJECT_ROOT manually."
    )


if MANUAL_PROJECT_ROOT is None:
    PROJECT_ROOT = find_project_root(Path.cwd())
else:
    PROJECT_ROOT = Path(MANUAL_PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "analysis" / "data"

ADMISSIONS_PATH = DATA_DIR / "Lumiere_Country_Full_Backup.csv"
CLUSTERS_PATH = DATA_DIR / "clustered_films_k6.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "text_numeric_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FEATURES_PATH = OUTPUT_DIR / f"tfidf_svd_features_{N_SVD_COMPONENTS}.csv"
OUTPUT_CONFIG_PATH = OUTPUT_DIR / f"tfidf_svd_config_{N_SVD_COMPONENTS}.json"

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

Project root: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action
Data dir: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\analysis\data
Output dir: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\text_numeric_features


In [4]:
df_admissions = pd.read_csv(ADMISSIONS_PATH)
df_clusters = pd.read_csv(CLUSTERS_PATH)

print("Admissions shape:", df_admissions.shape)
print("Clusters shape:", df_clusters.shape)

print("\nAdmissions columns:")
print(df_admissions.columns.tolist())

print("\nClusters columns:")
print(df_clusters.columns.tolist())

Admissions shape: (2000, 24)
Clusters shape: (2000, 16)

Admissions columns:
['titleId', 'imdbUrl', 'originalTitle', 'titleType', 'releaseYear', 'runtimeMinutes', 'isAdult', 'imdbRating', 'numberOfVotes', 'allCountries', 'mainCountry', 'allLanguages', 'firstLanguage', 'englishTitle', 'topFiveActors', 'directors', 'writers', 'plotShort', 'plotMedium', 'plotLong', 'genres', 'keywords', 'production', 'country_admissions']

Clusters columns:
['plotMedium', 'plotShort', 'genres', 'keywords', 'mainCountry', 'firstLanguage', 'plot_final', 'plot_source', 'has_plot', 'plot_clean', 'genres_list', 'keywords_list', 'keywords_filtered', 'mainCountry_clean', 'firstLanguage_clean', 'cluster']


In [5]:
selected_text_column = "plot_clean"

if len(df_admissions) != len(df_clusters):
    raise ValueError(
        "Admissions and clusters files must have the same number of rows "
        "because clusters file has no titleId."
    )

df_text = pd.DataFrame({
    "titleId": df_admissions["titleId"].values,
    "text_for_tfidf": df_clusters[selected_text_column].values,
})

df_text["text_for_tfidf"] = df_text["text_for_tfidf"].fillna("").astype(str)
df_text = df_text.drop_duplicates(subset=["titleId"]).copy()

print("Selected text column:", selected_text_column)
print("Text dataframe shape:", df_text.shape)
print(df_text.head())

Selected text column: plot_clean
Text dataframe shape: (2000, 2)
     titleId                                     text_for_tfidf
0  tt3042384  poland 1962 anna an orphan brought up by nuns ...
1  tt2486278  loosely based on the true events of a homeless...
2  tt1361313  louis a young teacher enamored of the age of f...
3  tt0222124  an outlandish science fiction comedy thriller ...
4  tt0421229  mrs palfrey dame joan plowright recently widow...


In [6]:
tfidf_svd_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF,
        ngram_range=TFIDF_NGRAM_RANGE,
        stop_words="english",
        sublinear_tf=True
    )),
    ("svd", TruncatedSVD(
        n_components=N_SVD_COMPONENTS,
        random_state=RANDOM_STATE
    ))
])

text_svd_matrix = tfidf_svd_pipeline.fit_transform(df_text["text_for_tfidf"])

svd_columns = [
    f"text_svd_{i:03d}"
    for i in range(N_SVD_COMPONENTS)
]

df_text_features = pd.DataFrame(
    text_svd_matrix,
    columns=svd_columns
)

df_text_features.insert(
    0,
    "titleId",
    df_text["titleId"].values
)

print("TF-IDF/SVD feature dataframe shape:", df_text_features.shape)
print("Explained variance ratio sum:", tfidf_svd_pipeline.named_steps["svd"].explained_variance_ratio_.sum().round(4))
print(df_text_features.head())

TF-IDF/SVD feature dataframe shape: (2000, 51)
Explained variance ratio sum: 0.0923
     titleId  text_svd_000  text_svd_001  text_svd_002  text_svd_003  \
0  tt3042384      0.066002     -0.003300      0.023320      0.016317   
1  tt2486278      0.076587      0.027156     -0.013315      0.031591   
2  tt1361313      0.157874     -0.037987      0.003520      0.023498   
3  tt0222124      0.070387      0.118115     -0.048262     -0.033025   
4  tt0421229      0.127832     -0.030611      0.022623      0.036680   

   text_svd_004  text_svd_005  text_svd_006  text_svd_007  text_svd_008  ...  \
0     -0.002539      0.019307      0.000908      0.015232     -0.033808  ...   
1     -0.007966      0.029328      0.062422      0.112532     -0.025923  ...   
2     -0.065637      0.026476      0.024817     -0.017844      0.074935  ...   
3      0.018257     -0.031821      0.022827      0.039777      0.051008  ...   
4     -0.048893      0.029632      0.011147     -0.031952     -0.020493  ...   

  

In [7]:
df_text_features.to_csv(OUTPUT_FEATURES_PATH, index=False)

config = {
    "n_svd_components": N_SVD_COMPONENTS,
    "random_state": RANDOM_STATE,
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "tfidf_min_df": TFIDF_MIN_DF,
    "tfidf_max_df": TFIDF_MAX_DF,
    "tfidf_ngram_range": TFIDF_NGRAM_RANGE,
    "selected_text_column": selected_text_column,
    "input_rows": int(len(df_text)),
    "output_rows": int(len(df_text_features)),
    "output_columns": int(df_text_features.shape[1]),
    "explained_variance_ratio_sum": float(
        tfidf_svd_pipeline.named_steps["svd"].explained_variance_ratio_.sum()
    ),
}

with open(OUTPUT_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved features to:", OUTPUT_FEATURES_PATH)
print("Saved config to:", OUTPUT_CONFIG_PATH)

Saved features to: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\text_numeric_features\tfidf_svd_features_50.csv
Saved config to: c:\Users\Georg\Desktop\DTU\1. FATA\FataProject\Analytics2Action\outputs\text_numeric_features\tfidf_svd_config_50.json


In [8]:
df_check = pd.read_csv(OUTPUT_FEATURES_PATH)

print("Saved feature file shape:", df_check.shape)
print("Columns:")
print(df_check.columns.tolist()[:10], "...", df_check.columns.tolist()[-5:])

print(df_check.head())

Saved feature file shape: (2000, 51)
Columns:
['titleId', 'text_svd_000', 'text_svd_001', 'text_svd_002', 'text_svd_003', 'text_svd_004', 'text_svd_005', 'text_svd_006', 'text_svd_007', 'text_svd_008'] ... ['text_svd_045', 'text_svd_046', 'text_svd_047', 'text_svd_048', 'text_svd_049']
     titleId  text_svd_000  text_svd_001  text_svd_002  text_svd_003  \
0  tt3042384      0.066002     -0.003300      0.023320      0.016317   
1  tt2486278      0.076587      0.027156     -0.013315      0.031591   
2  tt1361313      0.157874     -0.037987      0.003520      0.023498   
3  tt0222124      0.070387      0.118115     -0.048262     -0.033025   
4  tt0421229      0.127832     -0.030611      0.022623      0.036680   

   text_svd_004  text_svd_005  text_svd_006  text_svd_007  text_svd_008  ...  \
0     -0.002539      0.019307      0.000908      0.015232     -0.033808  ...   
1     -0.007966      0.029328      0.062422      0.112532     -0.025923  ...   
2     -0.065637      0.026476      0.024